In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# --- CNN用（オプション） ---
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    USE_CNN = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🧠 PyTorch利用可能 (device: {device})")
except ImportError:
    USE_CNN = False
    print("⚠️ PyTorchなし → CNNスキップ（3モデルで進行）")

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み中...")
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

# ベイスギ除外（異常値）
train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

# ============================================================
# 2. 波数インデックスの特定（考察の表をコードに反映）
# ============================================================
wavenumbers = np.array([float(c) for c in spec_cols])

# ┌──────────────────────────────────────────────┐
# │ 考察の表:                                      │
# │  水の O-H  → 5150 cm⁻¹ (1940nm), 6900 cm⁻¹   │
# │  木材 C-H  → 5900 cm⁻¹ (1700nm), 4300 cm⁻¹   │
# └──────────────────────────────────────────────┘
idx_5150 = np.argmin(np.abs(wavenumbers - 5150))  # 水 O-H 結合音
idx_6900 = np.argmin(np.abs(wavenumbers - 6900))  # 水 O-H 第1倍音
idx_5900 = np.argmin(np.abs(wavenumbers - 5900))  # セルロース C-H
idx_4300 = np.argmin(np.abs(wavenumbers - 4300))  # C-H + O-H 結合音

# 罠3対策：ピークシフト検出用の帯域範囲
water_band_5150 = np.where((wavenumbers >= 5000) & (wavenumbers <= 5300))[0]
water_band_6900 = np.where((wavenumbers >= 6700) & (wavenumbers <= 7100))[0]

print(f"📏 スペクトル次元数: {len(spec_cols)}")
print(f"🔬 5150cm⁻¹ → idx {idx_5150} (実値: {wavenumbers[idx_5150]:.1f})")
print(f"🔬 6900cm⁻¹ → idx {idx_6900} (実値: {wavenumbers[idx_6900]:.1f})")
print(f"🔬 5900cm⁻¹ → idx {idx_5900} (実値: {wavenumbers[idx_5900]:.1f})")
print(f"🔬 4300cm⁻¹ → idx {idx_4300} (実値: {wavenumbers[idx_4300]:.1f})")

# ============================================================
# 3. 前処理関数
# ============================================================

def apply_snv(X):
    """罠2対策: Standard Normal Variate — 散乱によるベースライン変動を補正"""
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

def apply_msc(X, ref_spectrum):
    """罠2対策: Multiplicative Scatter Correction — trainの平均スペクトルを基準"""
    X_msc = np.zeros_like(X)
    for i in range(X.shape[0]):
        coef = np.polyfit(ref_spectrum, X[i, :], 1)
        X_msc[i, :] = (X[i, :] - coef[1]) / (coef[0] + 1e-8)
    return X_msc

def apply_derivatives(X):
    """罠1対策: Savitzky-Golay微分 — ブロードなO-Hピークを分離"""
    d1 = savgol_filter(X, window_length=15, polyorder=2, deriv=1, axis=1)
    d2 = savgol_filter(X, window_length=11, polyorder=2, deriv=2, axis=1)
    
    return d1, d2


def extract_physics_features(X_raw, X_d2):
    """
    考察ベースの物理特徴量抽出（すべて1サンプル単独で計算可能）
    
    罠1対策: 二次微分後の比率で O-H 被りを軽減
    罠3対策: Max/Argmax でピークシフトを捉える
    """
    features = {}
    eps_ratio = 1e-4
    ratio_clip = 100.0

    def safe_ratio(num, den):
        # 分母が極小のときの比率発散を抑制（Foldごとの不安定化対策）
        den_safe = np.where(np.abs(den) < eps_ratio, eps_ratio, np.abs(den))
        return np.clip(num / den_safe, -ratio_clip, ratio_clip)
    
    # ─────────────────────────────────────────────
    # 【罠1対策】二次微分後の比率特徴量
    #   含水率 = 水の質量/木材の質量 に直接対応する設計
    # ─────────────────────────────────────────────
    
    # d2(5150) / d2(5900) → 水のO-H / セルロースC-H ≈ 含水率
    d2_5150 = X_d2[:, idx_5150]
    d2_5900 = X_d2[:, idx_5900]
    d2_6900 = X_d2[:, idx_6900]
    d2_4300 = X_d2[:, idx_4300]
    
    features['ratio_d2_5150_5900'] = safe_ratio(d2_5150, d2_5900)
    features['ratio_d2_6900_5900'] = safe_ratio(d2_6900, d2_5900)
    features['ratio_d2_5150_4300'] = safe_ratio(d2_5150, d2_4300)
    
    # 生スペクトルの比率も残す（元コードの特徴量）
    features['ratio_raw_5150_5900'] = safe_ratio(X_raw[:, idx_5150], X_raw[:, idx_5900])
    features['ratio_raw_6900_5900'] = safe_ratio(X_raw[:, idx_6900], X_raw[:, idx_5900])
    
    # ─────────────────────────────────────────────
    # 【罠3対策】Max / Argmax 特徴量
    #   自由水→結合水の移行でピーク位置がシフトする現象を捉える
    # ─────────────────────────────────────────────
    
    # 5000-5300 cm⁻¹ 帯域（水の結合音）
    d2_water_5150 = np.abs(X_d2[:, water_band_5150])
    features['max_d2_water5150'] = np.max(d2_water_5150, axis=1)
    features['argmax_d2_water5150'] = np.argmax(d2_water_5150, axis=1).astype(float)
    
    # 実際の波数位置も特徴量に（argmaxをインデックス→波数に変換）
    features['peak_wn_water5150'] = wavenumbers[water_band_5150][
        np.argmax(d2_water_5150, axis=1)
    ]
    
    # 6700-7100 cm⁻¹ 帯域（水の第1倍音）
    d2_water_6900 = np.abs(X_d2[:, water_band_6900])
    features['max_d2_water6900'] = np.max(d2_water_6900, axis=1)
    features['argmax_d2_water6900'] = np.argmax(d2_water_6900, axis=1).astype(float)
    features['peak_wn_water6900'] = wavenumbers[water_band_6900][
        np.argmax(d2_water_6900, axis=1)
    ]
    
    # ─────────────────────────────────────────────
    # 【補助】散乱プロキシ（密度の間接指標）
    # ─────────────────────────────────────────────
    features['scatter_std'] = np.std(X_raw, axis=1)
    features['scatter_mean'] = np.mean(X_raw, axis=1)
    
    # ─────────────────────────────────────────────
    # 【補助】帯域ごとの積分値（面的な吸収量）
    # ─────────────────────────────────────────────
    features['area_water5150'] = np.trapezoid(np.abs(X_d2[:, water_band_5150]), axis=1)
    features['area_water6900'] = np.trapezoid(np.abs(X_d2[:, water_band_6900]), axis=1)
    
    return pd.DataFrame(features)


# ============================================================
# 4. 1D-CNN モデル定義（罠3のピークシフトを畳み込みで吸収）
# ============================================================

if USE_CNN:
    class NIR_CNN(nn.Module):
        """
        1D-CNN: 波形のズレ（並進移動）を畳み込みで自然に吸収する
        罠3のピークシフト対策として有効
        """
        def __init__(self, input_dim):
            super().__init__()
            self.conv_block = nn.Sequential(
                nn.Conv1d(1, 32, kernel_size=15, padding=7),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.MaxPool1d(4),
                nn.Dropout(0.2),
                
                nn.Conv1d(32, 64, kernel_size=11, padding=5),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.MaxPool1d(4),
                nn.Dropout(0.2),
                
                nn.Conv1d(64, 128, kernel_size=7, padding=3),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(8),
                nn.Dropout(0.3),
            )
            self.fc = nn.Sequential(
                nn.Linear(128 * 8, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 1)
            )
        
        def forward(self, x):
            x = x.unsqueeze(1)  # (batch, 1, wavelength)
            x = self.conv_block(x)
            x = x.view(x.size(0), -1)
            return self.fc(x).squeeze(-1)
    
    def train_cnn(X_tr, y_tr, X_va, y_va, input_dim, epochs=150, lr=1e-3):
        """CNNの学習ループ"""
        model = NIR_CNN(input_dim).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.MSELoss()
        
        train_ds = TensorDataset(
            torch.FloatTensor(X_tr).to(device),
            torch.FloatTensor(y_tr).to(device)
        )
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
        
        best_val_loss = float('inf')
        best_state = None
        patience_counter = 0
        
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
            scheduler.step()
            
            model.eval()
            with torch.no_grad():
                val_pred = model(torch.FloatTensor(X_va).to(device))
                val_loss = criterion(val_pred, torch.FloatTensor(y_va).to(device)).item()
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 20:
                    break
        
        model.load_state_dict(best_state)
        model.eval()
        return model


# ============================================================
# 5. メインCVループ（全処理をFold内で実施 → データリーク防止）
# ============================================================

print("\n" + "=" * 60)
print("🚀 考察ベース改良モデル — CVループ開始")
print("=" * 60)

gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

# Fold平均用の蓄積配列
final_lgb = np.zeros(len(test))
final_pls = np.zeros(len(test))
final_rdg = np.zeros(len(test))
final_cnn = np.zeros(len(test)) if USE_CNN else None

oof_preds = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    print(f"\n{'─'*50}")
    print(f"📁 Fold {fold+1}/5")
    print(f"{'─'*50}")
    
    # ── 生データ分割 ──
    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()
    
    # ── 罠2対策: SNV ──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)
    
    # ── 罠2対策: MSC (trainの平均スペクトルを基準) ──
    ref_spectrum = np.mean(X_tr_raw, axis=0)  # trainのみで計算 → ルール準拠
    msc_tr = apply_msc(X_tr_raw, ref_spectrum)
    msc_va = apply_msc(X_va_raw, ref_spectrum)
    msc_te = apply_msc(X_te_raw, ref_spectrum)
    
    # ── 罠1対策: Savitzky-Golay微分（SNV後に適用） ──
    d1_tr, d2_tr = apply_derivatives(snv_tr)
    d1_va, d2_va = apply_derivatives(snv_va)
    d1_te, d2_te = apply_derivatives(snv_te)
    
    # ── MSCに対しても微分 ──
    _, d2_msc_tr = apply_derivatives(msc_tr)
    _, d2_msc_va = apply_derivatives(msc_va)
    _, d2_msc_te = apply_derivatives(msc_te)
    
    # ── 罠1&3対策: 物理特徴量の抽出 ──
    phys_tr = extract_physics_features(X_tr_raw, d2_tr)
    phys_va = extract_physics_features(X_va_raw, d2_va)
    phys_te = extract_physics_features(X_te_raw, d2_te)
    
    # MSCベースの物理特徴量も追加
    phys_msc_tr = extract_physics_features(X_tr_raw, d2_msc_tr)
    phys_msc_va = extract_physics_features(X_va_raw, d2_msc_va)
    phys_msc_te = extract_physics_features(X_te_raw, d2_msc_te)
    phys_msc_tr.columns = ['msc_' + c for c in phys_msc_tr.columns]
    phys_msc_va.columns = ['msc_' + c for c in phys_msc_va.columns]
    phys_msc_te.columns = ['msc_' + c for c in phys_msc_te.columns]
    
    # ── PCA: trainのみでfit ──
    pca = PCA(n_components=15, random_state=42)
    pca_tr = pca.fit_transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)
    
    # ── KNN特徴量: PCA空間で近傍のy平均＆距離 ──
    knn = NearestNeighbors(n_neighbors=10, metric='cosine')
    knn.fit(pca_tr)
    
    # train自身（自分を除外するため k+1）
    dist_tr, ind_tr = knn.kneighbors(pca_tr, n_neighbors=11)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)
    knn_ystd_tr  = np.std(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)
    knn_dist_tr  = np.mean(dist_tr[:, 1:], axis=1).reshape(-1, 1)
    
    # validation
    dist_va, ind_va = knn.kneighbors(pca_va, n_neighbors=10)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
    knn_ystd_va  = np.std(y_tr[ind_va], axis=1).reshape(-1, 1)
    knn_dist_va  = np.mean(dist_va, axis=1).reshape(-1, 1)
    
    # test
    dist_te, ind_te = knn.kneighbors(pca_te, n_neighbors=10)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)
    knn_ystd_te  = np.std(y_tr[ind_te], axis=1).reshape(-1, 1)
    knn_dist_te  = np.mean(dist_te, axis=1).reshape(-1, 1)
    
    # ───────────────────────────────────────────
    # モデル1: LightGBM（全部のせ）
    # ───────────────────────────────────────────
    feat_tr_lgb = np.hstack([
        snv_tr, d1_tr, pca_tr,
        knn_ymean_tr, knn_ystd_tr, knn_dist_tr,
        phys_tr.values, phys_msc_tr.values
    ])
    feat_va_lgb = np.hstack([
        snv_va, d1_va, pca_va,
        knn_ymean_va, knn_ystd_va, knn_dist_va,
        phys_va.values, phys_msc_va.values
    ])
    feat_te_lgb = np.hstack([
        snv_te, d1_te, pca_te,
        knn_ymean_te, knn_ystd_te, knn_dist_te,
        phys_te.values, phys_msc_te.values
    ])
    
    lgb_model = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.02, max_depth=5,
        num_leaves=31, subsample=0.7, colsample_bytree=0.25,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr_lgb, y_tr,
        eval_set=[(feat_va_lgb, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    p_va_lgb = np.expm1(lgb_model.predict(feat_va_lgb))
    p_te_lgb = np.expm1(lgb_model.predict(feat_te_lgb))
    
    # ───────────────────────────────────────────
    # モデル2: PLS回帰（二次微分のみ — 化学計測の王道）
    # ───────────────────────────────────────────
    pls_model = PLSRegression(n_components=8)
    pls_model.fit(d2_tr, y_tr)
    p_va_pls = np.expm1(pls_model.predict(d2_va).flatten())
    p_te_pls = np.expm1(pls_model.predict(d2_te).flatten())
    
    # ───────────────────────────────────────────
    # モデル3: Ridge回帰（圧縮特徴量 + 物理特徴量）
    # ───────────────────────────────────────────
    feat_tr_rdg = np.hstack([
        pca_tr, knn_ymean_tr, knn_ystd_tr, knn_dist_tr,
        phys_tr.values
    ])
    feat_va_rdg = np.hstack([
        pca_va, knn_ymean_va, knn_ystd_va, knn_dist_va,
        phys_va.values
    ])
    feat_te_rdg = np.hstack([
        pca_te, knn_ymean_te, knn_ystd_te, knn_dist_te,
        phys_te.values
    ])
    
    # Ridgeにはスケーリングが必要
    # train分布に基づくクリップで外れ値の影響を抑制
    rdg_clip_lo = np.percentile(feat_tr_rdg, 1.0, axis=0)
    rdg_clip_hi = np.percentile(feat_tr_rdg, 99.0, axis=0)
    feat_tr_rdg = np.clip(feat_tr_rdg, rdg_clip_lo, rdg_clip_hi)
    feat_va_rdg = np.clip(feat_va_rdg, rdg_clip_lo, rdg_clip_hi)
    feat_te_rdg = np.clip(feat_te_rdg, rdg_clip_lo, rdg_clip_hi)
    
    scaler_rdg = StandardScaler()
    feat_tr_rdg_s = scaler_rdg.fit_transform(feat_tr_rdg)
    feat_va_rdg_s = scaler_rdg.transform(feat_va_rdg)
    feat_te_rdg_s = scaler_rdg.transform(feat_te_rdg)
    
    rdg_model = Ridge(alpha=10.0, random_state=42)
    rdg_model.fit(feat_tr_rdg_s, y_tr)
    p_va_rdg_log = rdg_model.predict(feat_va_rdg_s)
    p_te_rdg_log = rdg_model.predict(feat_te_rdg_s)
    p_va_rdg = np.expm1(np.clip(p_va_rdg_log, -2.0, 6.0))
    p_te_rdg = np.expm1(np.clip(p_te_rdg_log, -2.0, 6.0))
    
    # ───────────────────────────────────────────
    # モデル4: 1D-CNN（ピークシフトを畳み込みで吸収）
    # ───────────────────────────────────────────
    if USE_CNN:
        # CNNにはSNV後のスペクトルを入力
        scaler_cnn = StandardScaler()
        cnn_tr = scaler_cnn.fit_transform(snv_tr)
        cnn_va = scaler_cnn.transform(snv_va)
        cnn_te = scaler_cnn.transform(snv_te)
        
        cnn_model = train_cnn(
            cnn_tr, y_tr, cnn_va, y_va,
            input_dim=cnn_tr.shape[1], epochs=200, lr=1e-3
        )
        
        with torch.no_grad():
            p_va_cnn = np.expm1(
                cnn_model(torch.FloatTensor(cnn_va).to(device)).cpu().numpy()
            )
            p_te_cnn = np.expm1(
                cnn_model(torch.FloatTensor(cnn_te).to(device)).cpu().numpy()
            )
    
    # ───────────────────────────────────────────
    # ブレンド
    # ───────────────────────────────────────────
    if USE_CNN:
        # 4モデルブレンド: LGB 55%, PLS 15%, Ridge 15%, CNN 15%
        w_lgb, w_pls, w_rdg, w_cnn = 0.55, 0.15, 0.15, 0.15
        p_va_blend = (p_va_lgb * w_lgb + p_va_pls * w_pls
                      + p_va_rdg * w_rdg + p_va_cnn * w_cnn)
        p_te_blend = (p_te_lgb * w_lgb + p_te_pls * w_pls
                      + p_te_rdg * w_rdg + p_te_cnn * w_cnn)
    else:
        # 3モデルブレンド: LGB 65%, PLS 20%, Ridge 15%
        w_lgb, w_pls, w_rdg = 0.65, 0.20, 0.15
        p_va_blend = p_va_lgb * w_lgb + p_va_pls * w_pls + p_va_rdg * w_rdg
        p_te_blend = p_te_lgb * w_lgb + p_te_pls * w_pls + p_te_rdg * w_rdg
    
    # Fold蓄積
    final_lgb += p_te_lgb / 5
    final_pls += p_te_pls / 5
    final_rdg += p_te_rdg / 5
    if USE_CNN:
        final_cnn += p_te_cnn / 5
    
    oof_preds[va_idx] = p_va_blend
    
    # 個別RMSE
    rmse_lgb = np.sqrt(mean_squared_error(np.expm1(y_va), p_va_lgb))
    rmse_pls = np.sqrt(mean_squared_error(np.expm1(y_va), p_va_pls))
    rmse_rdg = np.sqrt(mean_squared_error(np.expm1(y_va), p_va_rdg))
    rmse_blend = np.sqrt(mean_squared_error(np.expm1(y_va), p_va_blend))
    fold_rmses.append(rmse_blend)
    
    print(f"  LGB   RMSE: {rmse_lgb:.4f}")
    print(f"  PLS   RMSE: {rmse_pls:.4f}")
    print(f"  Ridge RMSE: {rmse_rdg:.4f}")
    if USE_CNN:
        rmse_cnn = np.sqrt(mean_squared_error(np.expm1(y_va), p_va_cnn))
        print(f"  CNN   RMSE: {rmse_cnn:.4f}")
    print(f"  🌟 Blend RMSE: {rmse_blend:.4f}")


# ============================================================
# 6. 全体OOF評価
# ============================================================
oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_preds))
print(f"\n{'='*60}")
print(f"📊 全体 OOF RMSE: {oof_rmse:.4f}")
print(f"📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")
print(f"{'='*60}")


# ============================================================
# 7. 最終ブレンド & 提出ファイル
# ============================================================
if USE_CNN:
    final_blend = (final_lgb * w_lgb + final_pls * w_pls
                   + final_rdg * w_rdg + final_cnn * w_cnn)
else:
    final_blend = final_lgb * w_lgb + final_pls * w_pls + final_rdg * w_rdg

# 含水率は0%以上（負の値はありえない）
final_blend = np.clip(final_blend, 0, None)

submit[1] = final_blend
output_filename = 'submission_physics_informed.csv'
submit.to_csv(output_filename, index=False, header=False)

print(f"\n✅ 提出ファイル生成完了: {output_filename}")
print(f"📈 予測統計: min={final_blend.min():.1f}%, "
      f"median={np.median(final_blend):.1f}%, max={final_blend.max():.1f}%")

🧠 PyTorch利用可能 (device: cuda)
📂 データ読み込み中...
📏 スペクトル次元数: 1555
🔬 5150cm⁻¹ → idx 1256 (実値: 5149.2)
🔬 6900cm⁻¹ → idx 802 (実値: 6900.4)
🔬 5900cm⁻¹ → idx 1061 (実値: 5901.4)
🔬 4300cm⁻¹ → idx 1476 (実値: 4300.7)

🚀 考察ベース改良モデル — CVループ開始

──────────────────────────────────────────────────
📁 Fold 1/5
──────────────────────────────────────────────────
  LGB   RMSE: 9.6908
  PLS   RMSE: 12.1569
  Ridge RMSE: 19.8945
  CNN   RMSE: 13.6155
  🌟 Blend RMSE: 10.0843

──────────────────────────────────────────────────
📁 Fold 2/5
──────────────────────────────────────────────────
  LGB   RMSE: 20.5281
  PLS   RMSE: 14.7467
  Ridge RMSE: 21.3028
  CNN   RMSE: 31.3241
  🌟 Blend RMSE: 16.2245

──────────────────────────────────────────────────
📁 Fold 3/5
──────────────────────────────────────────────────
  LGB   RMSE: 22.0519
  PLS   RMSE: 9.8101
  Ridge RMSE: 22.7162
  CNN   RMSE: 17.8427
  🌟 Blend RMSE: 15.5083

──────────────────────────────────────────────────
📁 Fold 4/5
──────────────────────────────────────